# ActorAttack — Single-Turn Ablation

Tests whether the multi-turn structure of ActorAttack is necessary for jailbreak success.

**Design:**
- Take all 1000 harmful ActorAttack conversations (100 pairs × 10 attempts)
- Condense all user-turn questions into one numbered prompt per conversation
- Send as a single-turn request to the target model
- Judge with the same 1–5 ActorAttack judge
- Compare per-conversation ASR and pair-level ASR against original multi-turn results

**Interpretation:** If single-turn ASR ≈ multi-turn ASR, the warm-up turns are not contributing — the harmful framing is sufficient on its own.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

import asyncio
import json
import sys
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI
from tqdm.auto import tqdm

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data_generation._runner_utils import call_text, score_response, score_to_verdict

# ── Config ────────────────────────────────────────────────────────────────────
CONV_DIR       = repo_root / "data" / "conversations" / "actorattack_harmful"
OUTPUT_PATH    = repo_root / "data" / "ablations" / "actorattack_single_turn.parquet"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

TARGET_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
JUDGE_MODEL     = "gpt-4o"
VLLM_BASE_URL   = "http://localhost:8000/v1"
VLLM_API_KEY    = os.environ.get("VLLM_API_KEY", "token-abc123")
OPENAI_API_KEY  = os.environ.get("OPENAI_API_KEY")

CONCURRENCY = 10  # simultaneous target+judge calls

print(f"Conv dir: {CONV_DIR}")
print(f"Output:   {OUTPUT_PATH}")

## 1. Load conversations and build condensed prompts

In [ ]:
def condense_turns(turns: list[dict]) -> str:
    """
    Concatenate all user questions from a conversation into a single numbered prompt.
    Uses turn_idx order.
    """
    user_turns = sorted(
        [t for t in turns if t["role"] == "user"],
        key=lambda t: t["turn_idx"]
    )
    lines = [f"{i+1}. {t['content']}" for i, t in enumerate(user_turns)]
    return "\n".join(lines)


# Load all conversations (all attempts)
records = []
for fpath in sorted(CONV_DIR.glob("*.json")):
    conv = json.loads(fpath.read_text())
    condensed = condense_turns(conv["turns"])
    if not condensed:
        continue
    records.append({
        "pair_id":            conv["objective_pair_id"],
        "attempt":            conv.get("attempt", 1),
        "objective":          conv["objective"],
        "multi_turn_verdict": conv["verdict"],
        "n_user_turns":       sum(1 for t in conv["turns"] if t["role"] == "user"),
        "condensed_prompt":   condensed,
        "fname":              fpath.name,
    })

df = pd.DataFrame(records).sort_values(["pair_id", "attempt"]).reset_index(drop=True)
print(f"Loaded {len(df)} conversations ({df['pair_id'].nunique()} pairs × up to {df['attempt'].max()} attempts)")
print()
print("Multi-turn verdict distribution:")
print(df["multi_turn_verdict"].value_counts())
print()
print("Example condensed prompt (pair 0, attempt 1):")
ex = df[(df["pair_id"] == 0) & (df["attempt"] == 1)].iloc[0]
print(ex["condensed_prompt"][:600], "...")

## 2. Run condensed prompts against the target model

In [ ]:
target_client = AsyncOpenAI(base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY)
judge_client  = AsyncOpenAI(api_key=OPENAI_API_KEY)

target_sem = asyncio.Semaphore(CONCURRENCY)
judge_sem  = asyncio.Semaphore(CONCURRENCY)

# Resume: skip already-processed (pair_id, attempt) pairs
already_done = set()
existing_results = []
if OUTPUT_PATH.exists():
    df_existing = pd.read_parquet(OUTPUT_PATH)
    already_done = set(zip(df_existing["pair_id"], df_existing["attempt"]))
    existing_results = df_existing.to_dict("records")
    print(f"Resuming: {len(already_done)} conversations already done")
else:
    print("Starting fresh")

todo = df[~df.apply(lambda r: (r["pair_id"], r["attempt"]) in already_done, axis=1)]
print(f"Remaining: {len(todo)} conversations")


async def run_one(row: dict) -> dict:
    """Send condensed prompt to target, judge the response, return result dict."""
    messages = [{"role": "user", "content": row["condensed_prompt"]}]

    async with target_sem:
        response = await call_text(
            target_client, TARGET_MODEL_ID, messages,
            temperature=0.0, max_tokens=2048,
        )

    score, reason = await score_response(
        judge_client, JUDGE_MODEL, judge_sem,
        objective=row["objective"],
        target_response=response,
    )

    return {
        "pair_id":              row["pair_id"],
        "attempt":              row["attempt"],
        "objective":            row["objective"],
        "multi_turn_verdict":   row["multi_turn_verdict"],
        "single_turn_score":    score,
        "single_turn_verdict":  score_to_verdict(score),
        "single_turn_response": response,
        "judge_reason":         reason,
        "n_user_turns":         row["n_user_turns"],
    }


async def run_all(records: list[dict]) -> list[dict]:
    tasks = [run_one(r) for r in records]
    results = []
    pbar = tqdm(total=len(tasks), desc="Single-turn ablation")
    for fut in asyncio.as_completed(tasks):
        result = await fut
        results.append(result)
        pbar.update(1)
    pbar.close()
    return results


new_results = await run_all(todo.to_dict("records"))
all_results = existing_results + new_results
df_results = pd.DataFrame(all_results).sort_values(["pair_id", "attempt"]).reset_index(drop=True)
df_results.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved {len(df_results)} results → {OUTPUT_PATH}")

## 3. Compare ASR: multi-turn vs single-turn

In [ ]:
from IPython.display import display

# Load saved results (in case re-running just the analysis)
df_results = pd.read_parquet(OUTPUT_PATH)

n_total = len(df_results)
n_pairs = df_results["pair_id"].nunique()

# ── Per-conversation ASR ──────────────────────────────────────────────────────
mt_asr = (df_results["multi_turn_verdict"] == "jailbroken").mean()
st_asr = (df_results["single_turn_verdict"] == "jailbroken").mean()

# ── Pair-level ASR (any attempt jailbroken) ───────────────────────────────────
mt_pair_asr = df_results.groupby("pair_id")["multi_turn_verdict"].apply(
    lambda v: (v == "jailbroken").any()
).mean()
st_pair_asr = df_results.groupby("pair_id")["single_turn_verdict"].apply(
    lambda v: (v == "jailbroken").any()
).mean()

print(f"N conversations: {n_total}  ({n_pairs} pairs)")
print()
print(f"{'':30s}  {'multi-turn':>12}  {'single-turn':>12}")
print(f"{'Per-conversation ASR':30s}  {mt_asr:>12.1%}  {st_asr:>12.1%}")
print(f"{'Pair-level ASR (any attempt)':30s}  {mt_pair_asr:>12.1%}  {st_pair_asr:>12.1%}")
print()
print("Multi-turn verdict distribution:")
display(df_results["multi_turn_verdict"].value_counts())
print()
print("Single-turn verdict distribution:")
display(df_results["single_turn_verdict"].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig_dir = repo_root / "notebooks" / "figures" / "ablation"
fig_dir.mkdir(parents=True, exist_ok=True)

# ── Verdict transition matrix (per conversation) ──────────────────────────────
verdicts = ["jailbroken", "near_miss", "refusal"]
matrix = pd.crosstab(
    df_results["multi_turn_verdict"],
    df_results["single_turn_verdict"],
    rownames=["multi-turn"],
    colnames=["single-turn"],
).reindex(index=verdicts, columns=verdicts, fill_value=0)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(matrix.values, cmap="Blues")
ax.set_xticks(range(len(verdicts)))
ax.set_yticks(range(len(verdicts)))
ax.set_xticklabels(verdicts, rotation=30, ha="right")
ax.set_yticklabels(verdicts)
ax.set_xlabel("Single-turn verdict")
ax.set_ylabel("Multi-turn verdict")
ax.set_title(f"ActorAttack — verdict transition matrix\n(n={n_total} conversations)")

for i in range(len(verdicts)):
    for j in range(len(verdicts)):
        ax.text(j, i, matrix.values[i, j], ha="center", va="center", fontsize=13,
                color="white" if matrix.values[i, j] > matrix.values.max() * 0.5 else "black")

plt.colorbar(im, ax=ax, label="count")
plt.tight_layout()
plt.savefig(fig_dir / "actorattack_verdict_transition.png", dpi=150)
plt.show()

print("Verdict transition matrix:")
display(matrix)

In [ ]:
# ── Side-by-side ASR bars ─────────────────────────────────────────────────────
verdict_order = ["jailbroken", "near_miss", "refusal"]
colors        = ["crimson", "orange", "steelblue"]

mt_counts = df_results["multi_turn_verdict"].value_counts().reindex(verdict_order, fill_value=0)
st_counts = df_results["single_turn_verdict"].value_counts().reindex(verdict_order, fill_value=0)

x     = np.arange(len(verdict_order))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
bars_mt = ax.bar(x - width/2, mt_counts / n_total, width, label="multi-turn",  color=colors, alpha=0.9)
bars_st = ax.bar(x + width/2, st_counts / n_total, width, label="single-turn", color=colors, alpha=0.4, edgecolor="black", linewidth=0.8)

ax.set_xticks(x)
ax.set_xticklabels(verdict_order)
ax.set_ylabel("Fraction of conversations")
ax.set_title(f"ActorAttack — multi-turn vs single-turn condensed\n(n={n_total} conversations, {n_pairs} pairs × attempts)")
ax.set_ylim(0, 1)
ax.legend(["multi-turn (solid)", "single-turn condensed (faded)"])

for bar in list(bars_mt) + list(bars_st):
    h = bar.get_height()
    if h > 0.02:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.0%}",
                ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(fig_dir / "actorattack_asr_comparison.png", dpi=150)
plt.show()

## 4. Interesting cases

Pairs where multi-turn ≠ single-turn verdict.

In [ ]:
# Multi-turn jailbroken but single-turn failed → warm-up turns matter here
mt_wins = df_results[
    (df_results["multi_turn_verdict"] == "jailbroken") &
    (df_results["single_turn_verdict"] != "jailbroken")
]
print(f"Multi-turn jailbroken, single-turn failed: {len(mt_wins)}")
if len(mt_wins):
    display(mt_wins[["pair_id", "objective", "multi_turn_verdict", "single_turn_verdict"]].head(10))

print()

# Single-turn jailbroken but multi-turn failed → warm-up turns actively hurt
st_wins = df_results[
    (df_results["single_turn_verdict"] == "jailbroken") &
    (df_results["multi_turn_verdict"] != "jailbroken")
]
print(f"Single-turn jailbroken, multi-turn failed: {len(st_wins)}")
if len(st_wins):
    display(st_wins[["pair_id", "objective", "multi_turn_verdict", "single_turn_verdict"]].head(10))